In [ ]:
# !python -m pip install pyyaml==5.1
!pip install pyyaml
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities.
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

In [ ]:
import os

# os.environ["HF_TOKEN"] = "[Insert your HugginFace Token]"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)
os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNNMedicalCustomTimm(nn.Module):
    def __init__(self, model_base, model_weights, layer=[512, 256], dropout_rate=0.4, discriminative_fine=True):
        super().__init__()
        self.device = device

        # 1. Carrega o modelo de visão computacional puro sangue
        self.backbone = timm.create_model(
            "convnextv2_base.fcmae_ft_in22k_in1k",
            pretrained=True,
            num_classes=0,
        )

        data_config = timm.data.resolve_model_data_config(self.backbone)
        self.input_size = data_config["input_size"]      # (3, 224, 224)
        self.mean = torch.tensor(data_config["mean"]).view(1, 3, 1, 1)
        self.std = torch.tensor(data_config["std"]).view(1, 3, 1, 1)

        output_features = self.backbone.num_features

        if (discriminative_fine == True):
            for p in self.backbone.parameters():
                p.requires_grad = False
        
            for p in self.backbone.stages[-1].parameters():
                p.requires_grad = True
        else:
            for p in self.backbone.parameters():
                p.requires_grad = True

        self.layers = nn.ModuleList()
        self.norm = nn.ModuleList()
        self.activations = nn.ModuleList()
        self.dropout = nn.Dropout(p=dropout_rate) if dropout_rate != 0 else None

        if (len(layer) != 0):
            for i, neurons in enumerate(layer):
                in_f = output_features if i == 0 else layer[i-1]
                self.layers.append(nn.Linear(in_f, neurons))
                self.norm.append(nn.BatchNorm1d(neurons))
                self.activations.append(nn.GELU())
            self.output = nn.Linear(layer[-1], 1)
        else:
            self.output = nn.Linear(output_features, 1)

    def preprocess_batch(self, x):
        x = x.to(self.device, non_blocking=True)

        if x.dtype != torch.float32:
            x = x.float()

        if x.max() > 1.0:
            x = x / 255.0

        _, h, w = self.input_size
        if x.shape[-2:] != (h, w):
            x = F.interpolate(x, size=(h, w), mode="bilinear", align_corners=False)

        mean = self.mean.to(x.device)
        std = self.std.to(x.device)
        x = (x - mean) / std
        return x


    def forward(self, x):
        x = self.preprocess_batch(x)

        x = self.backbone(x)

        # Head
        for layer, norm, activation in zip(self.layers, self.norm, self.activations):
            x = activation(norm(layer(x)))
            if self.dropout is not None:
                x = self.dropout(x)

        output = self.output(x)
        return output

In [ ]:
import os
import cv2
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2 import model_zoo
import torch

# -------------------------------------------------------------------------------------------------------
# Detectron2

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
assert device == "cuda", "Running on CPU. A CUDA-enabled device is highly recommended. If you must run on CPU, comment out this line."

# carregando pesos já treinados
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")) # arquivo baixado direto do repositório detectron2
cfg.MODEL.WEIGHTS = "./TreinamentoSegmentacao/Segmenta_model_final.pth"  # caminho p/ modelo
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7 # para o modelo só mostrar detecções com confiança acima de 70%
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1 # o modelo só precisa detectar e segmentar uma classe, nesse caso o disco óptico
cfg.MODEL.DEVICE = "cpu" 

predictor = DefaultPredictor(cfg) # carrega o modelo

# TRAIN091555, RG
# TRAIN090004, NRG
img = cv2.imread("./TRAIN091555.jpg") # carrega a imagem para classificação
outputs = predictor(img)

# preprando informações das caixas delimitadoraas
v = Visualizer(img[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
instances = outputs["instances"].to("cpu")
boxes = instances.pred_boxes
scores = instances.scores
class_ids = instances.pred_classes

bbox = boxes.tensor[0].numpy()
score = scores[0].item()
class_id = class_ids[0].item()

# instruções oara recortar o disco
x1, y1, x2, y2 = map(int, bbox)
cropped_img = img[y1:y2, x1:x2]
cropped_img = cv2.resize(cropped_img, (390, 390))

# -------------------------------------------------------------------------------------------------------
# ConvNextV2 + MLP

import matplotlib.pyplot as plt
import numpy as np
from torchvision.models import convnext_base, ConvNeXt_Base_Weights
import torch
import torchvision.transforms.functional as TF

img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB)
img = torch.from_numpy(img).permute(2, 0, 1).contiguous().unsqueeze(0).float()

name_model = "convnextv2_base.fcmae_ft_in22k_in1_29-04-26_19-52-01_e30_oAdamW_lr0.001_b10_discos_15371_comAG2_Normal"

load_model = CNNMedicalCustomTimm(model_base=convnext_base, model_weights=ConvNeXt_Base_Weights.DEFAULT, layer=[256], dropout_rate=0.3, discriminative_fine=True).to(device)
state_dict = torch.load(f'./dados/Pytorch/ODJ/{name_model}/{name_model}.pth', weights_only=True, map_location=device)
load_model.load_state_dict(state_dict)

custom_threshold = 0.65
enable_tta = True

load_model.eval()
with torch.no_grad():
    ## TTA
    if (enable_tta == False):
        logits = load_model(img)
    else:
        logits_orig = load_model(img)

        img_hflip = TF.hflip(img)
        logits_hflip = load_model(img_hflip)

        img_vflip = TF.vflip(img)
        logits_vflip = load_model(img_vflip)

        logits = (logits_orig + logits_hflip + logits_vflip) / 3.0
        
    prob = torch.sigmoid(logits).squeeze().item()
    pred = int(prob >= custom_threshold)
del load_model

pred = int(prob >= custom_threshold)
classe = "RG" if pred == 1 else "NRG"
resultado = f'{classe} | score={prob:.4f} | thr={custom_threshold:.2f}'

v.draw_box(bbox, edge_color="green", alpha=score) # desenha a localização correta do disco óptico
v.draw_text(resultado, position=((x1+x2)/2,y1 - 100), color="green", horizontal_alignment="center", font_size=32) # desenha a probabilidade
out_img = v.get_output().get_image()

## Caso queira salvar a imagem completa.
cv2.imwrite("./output.png", cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR))

## Mostra a imagem utilizando Matplotlib
print(resultado)
plt.imshow(out_img)
plt.axis('off')
plt.show()